# **Abstract Syntax Tree**

In [2]:
import ast

In [3]:
eight = ast.Constant(8)

In [4]:
neg_eight = ast.UnaryOp(ast.USub(), eight)

In [5]:
read = ast.Call(ast.Name('input_int'),[])

In [6]:
simple_ast = ast.BinOp(read, ast.Add(), neg_eight)

In [7]:
ast_test_code = """
print(5+6)
20 - input_int(4)
"""
ast_parsed = ast.parse(ast_test_code)

In [8]:
print(ast.dump(ast_parsed,indent=4))

Module(
    body=[
        Expr(
            value=Call(
                func=Name(id='print', ctx=Load()),
                args=[
                    BinOp(
                        left=Constant(value=5),
                        op=Add(),
                        right=Constant(value=6))])),
        Expr(
            value=BinOp(
                left=Constant(value=20),
                op=Sub(),
                right=Call(
                    func=Name(id='input_int', ctx=Load()),
                    args=[
                        Constant(value=4)])))])


In [9]:
match simple_ast:
    case ast.BinOp(child1 , op, child2):
        print(op)

+


In [10]:
def leaf(arith):
    match arith:
        case ast.Constant(n):
            return True
        case ast.Call(ast.Name('input_int'),[]):
            return True
        case ast.UnaryOp(ast.USub(),e1):
            return False
        case ast.BinOp(e1,ast.Add(),e2):
            return False
        case ast.BinOp(e1,ast.Sub(),e2):
            return False
        case _:
            raise Exception('unexpected node')

In [11]:
print(leaf(ast.Call(ast.Name('input_int'),[])))

True


In [12]:
print(leaf(ast.UnaryOp(ast.USub(),eight)))

False


In [13]:
print(leaf(ast.Constant(eight)))

True


In [14]:
try:
    print(leaf(ast.BinOp(eight, ast.Mult(), eight)))
except Exception:
    print('error! unexpected node')

error! unexpected node


In [15]:
def is_exp(e):
    match e:
        case ast.Constant(n):
            return True
        case ast.Call(ast.Name('input_int'),[]):
            return True
        case ast.UnaryOp(ast.USub(),e1):
            return is_exp(e1)
        case ast.BinOp(e1, ast.Add(), e2):
            return is_exp(e1) and is_exp(e2)
        case ast.BinOp(e1,ast.Sub(), e2):
            return is_exp(e1) and is_exp(e2)
        case _:
            return False

In [16]:
def is_stmt(s):
    match s:
        case ast.Expr(ast.Call(ast.Name('input_int'),[e])):
            return is_exp(e)
        case ast.Expr(e):
            return is_exp(e)
        case _:
            return False

In [17]:
def is_Lint(p):
    match p:
        case ast.Module(body):
            return all([is_stmt(s) for s in body])
        case _:
            return False

In [18]:
print(is_Lint(ast.Module([ast.Expr(simple_ast)])))

True


In [19]:
print(is_Lint(ast.Module([ast.Expr(ast.BinOp(read,ast.Sub(),ast.UnaryOp(ast.Add(),eight)))])))

False


In [20]:
from utils import *

In [21]:
def interp_exp(e):
    match e:
        case ast.BinOp(left, ast.Add(), right):
            l = interp_exp(left)
            r = interp_exp(right)
            return add64(l,r)
        case ast.BinOp(left,ast.Sub(),right):
            l = interp_exp(left)
            r = interp_exp(right)
            return sub64(l,r)
        case ast.UnaryOp(ast.USub(),v):
            return neg64(interp_exp(v))
        case ast.Constant(value):
            return value
        case ast.Call(ast.Name('input_int'),[]):
            return input_int()

In [22]:
def interp_stmt(s):
    match s:
        case ast.Expr(ast.Call(ast.Name('print'),[arg])):
            print(interp_exp(arg))
        case ast.Expr(value):
            interp_exp(value)

In [23]:
def interp_Lint(p):
    match p:
        case ast.Module(body):
            for s in body:
                interp_stmt(s)

In [24]:
simple_program = """
print(10 + 3)"""

In [25]:
simple_program_ast = ast.parse(simple_program)

In [26]:
print(ast.dump(simple_program_ast,indent = 4))

Module(
    body=[
        Expr(
            value=Call(
                func=Name(id='print', ctx=Load()),
                args=[
                    BinOp(
                        left=Constant(value=10),
                        op=Add(),
                        right=Constant(value=3))]))])


In [27]:
simple_program_test = interp_Lint(simple_program_ast)

13


In [28]:
another_example_program = """
print(10+2)
print(20 + input_int())"""

another_example_program_ast = ast.parse(another_example_program)

In [30]:
print(ast.dump(another_example_program_ast,indent=4))

Module(
    body=[
        Expr(
            value=Call(
                func=Name(id='print', ctx=Load()),
                args=[
                    BinOp(
                        left=Constant(value=10),
                        op=Add(),
                        right=Constant(value=2))])),
        Expr(
            value=Call(
                func=Name(id='print', ctx=Load()),
                args=[
                    BinOp(
                        left=Constant(value=20),
                        op=Add(),
                        right=Call(
                            func=Name(id='input_int', ctx=Load())))]))])


In [31]:
another_example_program_test = interp_Lint(another_example_program_ast)

12
28
